<a href="https://colab.research.google.com/github/ncsu-geoforall-lab/GIS582-assignments/blob/main/5AB%20-%20Geomorphology/5A_Tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tutorial 5A: Geomorphometry I: Terrain Modeling

**Course:** [GIS 582 - Geospatial Modeling and Analysis](https://ncsu-geoforall-lab.github.io/geospatial-modeling-course/grass/terrain_modeling.html)  
**Institution:** [NC State University, Center for Geospatial Analytics](https://cnr.ncsu.edu/geospatial/)
**Instructors:** Helena Mitasova, Corey White, and team

## Learning Objectives

In this tutorial, you will learn how to:
- 3D mapping technologies: topography and bathymetry
- Digital elevation data representation
- Regular grid DEM and TIN-based elevation models
- Point cloud properties and analysis

## Tutorial Outline

* Part 1: Environment Setup
* Part 2: Analyze bare earth and multiple return lidar data properties by binning
* Part 3: Interpolate DTM and DSM
* Part 4: Optional: Visualize the interpolated DEMs in 3D with cutting planes

---
## Part 1: Environment Setup

### Install GRASS

**Important:** This setup takes 3-5 minutes. You'll need to run it each time you start a new Colab session.

In [ ]:
!add-apt-repository -y ppa:ubuntugis/ubuntugis-unstable
!apt update
!apt-get install -y grass-core grass-dev pdal

Check that GRASS and PDAL are installed by asking which versions are available.

In [ ]:
!grass --version
!pdal --version

Check which Python version is running.

In [ ]:
import sys

v = sys.version_info
print(f"We are using Python {v.major}.{v.minor}.{v.micro}")

### Import GRASS Python packages

We need to locate GRASS Python packages using `grass --config python_path` and add them to `sys.path`.

In [ ]:
import subprocess
import os
from pathlib import Path

# Ask GRASS where its Python packages are.
sys.path.append(
    subprocess.check_output(["grass", "--config", "python_path"], text=True).strip()
)

import grass.script as gs
import grass.jupyter as gj

import csv
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt

### Download North Carolina Sample Dataset

This dataset includes elevation, orthophotos, and sample lidar data used in this tutorial.

In [ ]:
!grass --tmp-project XY --exec g.download.project url=https://grass.osgeo.org/sampledata/north_carolina/nc_basic_spm_grass7.tar.gz path=/content

Download the LiDAR and orthophoto data for the area of interest. The data is in LAS format, which is a common format for storing LiDAR point cloud data.

Use the cell below to download the data directly into your Colab environment or access it through the links provided.

* [Folder with LAS file and orthophoto](https://drive.google.com/drive/folders/1C_vov2M85dOmHp7dzLLyW8JP--IrqonY?usp=drive_link)

Alternative links:

* [LAS tile 0793_016 in NC State Plane Meters](http://fatra.cnr.ncsu.edu/geospatial-modeling-course/data/tile_0793_016_spm.las)
* [Orthophoto geotif (mosaic at 1ft resolution)](http://fatra.cnr.ncsu.edu/geospatial-modeling-course/data/ortho_0793_016_ncspm.zip)

In [ ]:
!curl -L -o tile_0793_016_spm.las http://fatra.cnr.ncsu.edu/geospatial-modeling-course/data/tile_0793_016_spm.las
!curl -L -o ortho_0793_016_ncspm.zip http://fatra.cnr.ncsu.edu/geospatial-modeling-course/data/ortho_0793_016_ncspm.zip

### Initialize GRASS Session

In [ ]:
# Start GRASS session
grassdata = "/content"
project = "nc_basic_spm_grass7"
mapset = "user1"  # Create a new mapset for our work

# Start GRASS Session
session = gj.init(Path(project, mapset))
print("GRASS session started.")

In [ ]:
# Set computational region to elevation raster
gs.run_command("g.region", raster="elevation", flags="pg")

---
## Part 2: Analyze bare earth and multiple return lidar data properties by binning

Import the points using [v.in.pdal](https://grass.osgeo.org/grass-devel/manuals/v.in.pdal.html). We can specify which class and which return (first, middle, last) we want to import.
We can see the classification either in metadata distributed with the lidar data or it can be displayed with [pdal info](https://pdal.io/stages/info.html) command.

In [ ]:
!pdal info tile_0793_016_spm.las

Class 2 represents bare earth points.
Now we import bare earth points and first return points separately:

In [ ]:
gs.run_command(
    "v.in.pdal", 
    input="tile_0793_016_spm.las", 
    output="elev_lid793016_be", 
    class_filter=2
)
gs.run_command(
    "v.in.pdal", 
    input="tile_0793_016_spm.las", 
    output="elev_lid793016_1r", 
    return_filter="first"
)

Set the region from the imported point file with resolution of 1 meter.

In [ ]:
gs.run_command("g.region", vect="elev_lid793016_1r", res=1, flags="p")

### Point Density

Compute raster maps (with r.in.pdal) representing number of points per 1 m grid cell.

Compare point densities for bare earth, first return.

In [ ]:
gs.run_command(
    "r.in.pdal", 
    input="tile_0793_016_spm.las", 
    output="lid_be_binn1m",
    method="n", # number of points in cell
    input_srs="EPSG:4326",
    class_filter=2,
    flags="o",
    overwrite=True
)

gs.run_command(
    "r.in.pdal", 
    input="tile_0793_016_spm.las", 
    output="lid_1r_binn1m",
    method="n", # number of points in cell
    input_srs="EPSG:4326",
    return_filter="first", 
    flags="o",
    overwrite=True
)

# Set color scheme for the point density maps
gs.run_command("r.colors", map="lid_be_binn1m,lid_1r_binn1m", color="bcyr", flags="e")


In [ ]:
lidar_map_be = gj.Map(filename="be_pointdensity.png", height=450, width=450)
lidar_map_be.d_rast(map="lid_be_binn1m")
lidar_map_be.d_legend(raster="lid_be_binn1m", at=(2,50,2,9))
lidar_map_be.show()

In [ ]:
gs.parse_command("r.report", map="lid_be_binn1m", unit="p,c")

In [ ]:
gs.parse_command("r.univar", map="lid_be_binn1m")

In [ ]:
lidar_map_1r = gj.Map(filename="first_pointdensity.png", height=450, width=450)
lidar_map_1r.d_rast(map="lid_1r_binn1m")
lidar_map_1r.d_legend(raster="lid_1r_binn1m", at=(2,50,8,12), flags="ds")
lidar_map_1r.show()

In [ ]:
gs.parse_command("r.report", map="lid_1r_binn1m", unit="p,c")

In [ ]:
gs.parse_command("r.univar", map="lid_1r_binn1m")

#### Question 2.1

How do the point densities for bare earth and first return compare?

`Add text here with your answer to question 2.1`

Compute a raster map representing mean elevation for each 1m cell. It will have holes.

In [ ]:
gs.run_command(
    "r.in.pdal", 
    input="tile_0793_016_spm.las", 
    output="lid_be_binmean1m",
    method="mean", # number of points in cell
    input_srs="EPSG:4326",
    class_filter=2,
    flags="o",
    overwrite=True
)

gs.run_command(
    "r.in.pdal", 
    input="tile_0793_016_spm.las", 
    output="lid_1r_binmean1m",
    method="mean", # number of points in cell
    input_srs="EPSG:4326",
    return_filter="first", 
    flags="o",
    overwrite=True
)

gs.run_command("r.colors", map="lid_be_binmean1m,lid_1r_binmean1m", color="elevation")

In [ ]:
gs.parse_command("r.info", map="lid_1r_binmean1m")

In [ ]:
lid_be_binmean1m_map = gj.Map(filename="lidbemean1m.png")
lid_be_binmean1m_map.d_rast(map="lid_be_binmean1m")
lid_be_binmean1m_map.d_legend(raster="lid_be_binmean1m", at=(2,40,2,5))
lid_be_binmean1m_map.show()

In [ ]:
lid_1r_binmean1m_map = gj.Map(filename="lid1rmean1m.png")
lid_1r_binmean1m_map.d_rast(map="lid_1r_binmean1m")
lid_1r_binmean1m_map.d_legend(raster="lid_1r_binmean1m", at=(2,40,2,5))
lid_1r_binmean1m_map.show()

### Mean

Compute a raster map representing mean elevation for each 2m cell.
Result is almost good enough for 1st return, but there are still many holes for bare earth.

In [ ]:
gs.run_command("g.region", res=2, flags="ap")

gs.run_command("r.in.pdal", 
               input="tile_0793_016_spm.las", 
               output="lid_be_binmean2m",
               method="mean", # number of points in cell
               input_srs="EPSG:4326",
               class_filter=2,
               flags="o",
               overwrite=True)

gs.run_command("r.in.pdal", 
               input="tile_0793_016_spm.las", 
               output="lid_1r_binmean2m",
               method="mean", # number of points in cell
               input_srs="EPSG:4326",
               return_filter="first", 
               flags="o",
               overwrite=True)

gs.run_command("r.colors", map="lid_be_binmean2m,lid_1r_binmean2m", color="elevation")

In [ ]:
lid_be_binmean2m_map = gj.Map(filename="lid_be_binrange2m.png")
lid_be_binmean2m_map.d_rast(map="lid_be_binmean2m")
lid_be_binmean2m_map.d_legend(raster="lid_be_binmean2m", at=(2,40,2,5))
lid_be_binmean2m_map.show()


In [ ]:
lid_1r_binmean2m_map = gj.Map(filename="lid_1r_binrange2m.png")
lid_1r_binmean2m_map.d_rast(map="lid_1r_binmean2m")
lid_1r_binmean2m_map.d_legend(raster="lid_1r_binmean2m", at=(2,40,2,5))
lid_1r_binmean2m_map.show()

#### Question  2.2

Why are there holes in the binned DEMs? Why is the first return map mostly green (where is the orange-brown elevation?) Adjust the r.colors command for lid_1r_binmean2m that will redistribute the colors so that the topographic features are well represented. Hint, see assignment 2. Include the adjusted map in your report.

`Add text here with your answer`

In [ ]:
# Add map with adjusted color scheme here

### Range

Compute range of elevation values in each grid cell:

In [ ]:
gs.run_command(
    "r.in.pdal", 
    input="tile_0793_016_spm.las", 
    output="lid_be_binrange2m",
    method="range", # number of points in cell
    input_srs="EPSG:4326",
    class_filter=2,
    flags="o",
    overwrite=True
)

In [ ]:
lid_be_binrange2m_map = gj.Map(filename="lid_be_binrange2m.png")
lid_be_binrange2m_map.d_rast(map="lid_be_binrange2m")
lid_be_binrange2m_map.d_legend(raster="lid_be_binrange2m", at=(2,40,2,5))
lid_be_binrange2m_map.show()

In [ ]:
gs.run_command(
    "r.in.pdal", 
    input="tile_0793_016_spm.las", 
    output="lid_1r_binrange2m",
    method="range", # number of points in cell
    input_srs="EPSG:4326",
    return_filter="first", 
    flags="o",
    overwrite=True
)

In [ ]:
lid_1r_binrange2m_map = gj.Map(filename="lid_1r_binrange2m.png")
lid_1r_binrange2m_map.d_rast(map="lid_1r_binrange2m")
lid_1r_binrange2m_map.d_legend(raster="lid_1r_binrange2m", at=(2, 40, 2, 5))
lid_1r_binrange2m_map.show()

Import and display orthophoto, switch off all layers except for orthophoto.

In [ ]:
# Notice we are reading the zip file directly without unzipping it first.
# We could have also load the tif file directly without downloading and unzipping it first using the
# GDAL /vsicurl/vsizip/http://fatra.cnr.ncsu.edu/geospatial-modeling-course/data/ortho_0793_016_ncspm.zip/ortho_0793_016_ncspm.tif
gs.run_command(
    "r.in.gdal",
    input="/vsizip/ortho_0793_016_ncspm.zip/ortho_0793_016_ncspm.tif",
    output="ortho_2013_0793",
)

ortho_2013_0793_map = gj.Map()
ortho_2013_0793_map.d_rast(map="ortho_2013_0793")
ortho_2013_0793_map.show()

Identify the features that are associated with large range values.
Display only the high values of range.

In [ ]:
mylid_1rrange2m_map = gj.Map(filename="mylid_1rrange2m.png")
mylid_1rrange2m_map.d_rast(map="ortho_2013_0793")
mylid_1rrange2m_map.d_rast(map="lid_1r_binrange2m", values="10.-200")
mylid_1rrange2m_map.show()

#### Question 2.3

What landcover is associated with large range in multiple return data?

`Add text here with your answer to question 2.3`

We now know how dense the data are at 1m and 2m resolution and what is the range of elevation values within a grid cell.

If we need a 1m resolution DTM or DSM for our application this analysis tells us that the fast binning is not enough and we need to interpolate it from the point cloud.

---
## Part 3: Interpolate DTM and DSM

To interpolate DTM and DSM using the RST spline function we use default parameters except for number of points used for interpolation (npmin), minimum distance between points and higher tension and smoothing for multiple return data.

Be patient, it can take a few minutes to run depending on your computer power.

In [ ]:
gs.run_command("g.region", res=1, flags="ap")
gs.run_command(
    "v.surf.rst", 
    input="elev_lid793016_be", 
    elevation="elev_lid793016_be_1m",
    npmin=120,
    segmax=25,
    dmin=1
)

gs.run_command(
    "v.surf.rst", 
    input="elev_lid793016_1r", 
    elevation="elev_lid793016_1r_1m",
    npmin=120,
    segmax=25,
    tension=100,
    smooth=0.5,
    dmin=1
)

gs.run_command(
    "r.colors", map="elev_lid793016_be_1m,elev_lid793016_1r_1m", color="elevation"
)

Derive shaded relief for the interpolated DTM and DSM and display the result as colored shaded topography to highlight the terrain features.

In [ ]:
gs.run_command(
    "r.relief", input="elev_lid793016_be_1m", output="elev_lid793016_be_1msh"
)
gs.run_command(
    "r.relief", input="elev_lid793016_1r_1m", output="elev_lid793016_1r_1msh"
)

Display the maps with the elevation map in color and the shaded relief as a hillshade layer on top.

In [ ]:
elev_lid793016_be_1m_map = gj.Map(filename="elevation_be_1m.png")
elev_lid793016_be_1m_map.d_shade(shade="elev_lid793016_be_1msh", color="elev_lid793016_be_1m")
elev_lid793016_be_1m_map.d_legend(raster="elev_lid793016_be_1m", at=(2,40,2,5))
elev_lid793016_be_1m_map.show()

In [ ]:
elev_lid793016_1r_1m_map = gj.Map(filename="elevation_1r_1m.png")
elev_lid793016_1r_1m_map.d_shade(
    shade="elev_lid793016_1r_1msh", color="elev_lid793016_1r_1m"
)
elev_lid793016_1r_1m_map.d_legend(raster="elev_lid793016_1r_1m", at=(2, 40, 2, 5))
elev_lid793016_1r_1m_map.show()

#### Question 3.1

How does the binned first return DSM compare to interpolated first return DSM and what is the reason for the differences in range and coverage?

`Add text here with your answer to question 3.1`

#### Task 3.1

Compute a map of above ground height of vegetation and buildings from your interpolated bare ground and first return DEMs using map algebra. Include the output map with appropriate color ramp in your report. What is the mean height computed from this map?

In [ ]:
# Add code here for task 3.1

---
## Part 4: Optional: Visualize the interpolated DEMs in 3D with cutting planes

Use 3D view with cutting planes to compare the bare earth and terrain surface of the last 2 interpolated DEMs.

- elev_lid793016_1r_1m
- elev_lid793016_be_1m

Notes:

* Make sure fine resolution is set to 1 for all surfaces.
* Assign each surface constant color, add constant plane at 75m elevation for reference.
* Shade the crossection using the color by bottom surface option.
* If you don't remember this, see screen capture video for 3d view.
* Save image for your report.

In [ ]:
surface_3d_map = gj.Map3D()
surface_3d_map.render(
    elevation_map="elev_lid793016_be_1m",
    color_map="elev_lid793016_be_1m",
    perspective=20
)
surface_3d_map.overlay.d_legend(raster="elev_lid793016_be_1m", at=(60, 97, 87, 92))
surface_3d_map.show()

---
## Part 5: Optional - Visualize the point cloud in 3D

In [ ]:
!apt-get install -qq xvfb libgl1-mesa-glx
!pip install 'pyvista[all]' trame trame-vtk trame-vuetify laspy[lazrs] -qq

In [ ]:
import laspy
import pyvista as pv
import numpy as np

pv.set_jupyter_backend("html")
pv.global_theme.notebook = True  # otherwise nothing plots

# 1. Read the LAS file
las = laspy.read("/content/tile_0793_016_spm.las")

# 2. Extract coordinates (X, Y, Z)
points = np.stack((las.x, las.y, las.z), axis=1)

# 3. Create PyVista PolyData
cloud = pv.PolyData(points)

# 4. Add scalars like intensity or elevation
cloud["intensity"] = las.intensity

# 5. Visualize
pl = pv.Plotter()
pl.add_mesh(cloud, cmap="gist_earth", scalars="intensity", point_size=2)
pl.show()